# Phase 2: Handcrafted Feature Extraction + SVM

This notebook extends phase 1 with four additional handcrafted feature extraction methods:

- LBP
- Edge Histogram using Canny edges
- Hu Moments
- Gabor Features

The pipeline stays consistent with phase 1:

`image -> resize to 64x64 -> feature extraction -> StandardScaler -> SVM -> evaluate`

The existing `cropped_dataset/train`, `cropped_dataset/val`, and `cropped_dataset/test` folders are reused directly. No dataset files or phase 1 outputs are modified.

## Dependencies

If this notebook is run in a new environment, install the same package family used in phase 1:

```python
%pip install scikit-learn scikit-image opencv-python pandas numpy matplotlib joblib
```

In [1]:
from pathlib import Path
from time import perf_counter
import warnings

import cv2
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from skimage.feature import local_binary_pattern
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
try:
    from tqdm.auto import tqdm
except ModuleNotFoundError:
    def tqdm(iterable=None, **kwargs):
        return iterable if iterable is not None else []

warnings.filterwarnings('ignore', category=UserWarning)
plt.style.use('ggplot')

RANDOM_STATE = 42
DATASET_DIR = Path('cropped_dataset')
PHASE1_RESULTS_DIR = Path('log/exphase_1_result')
PHASE1_METADATA_PATH = PHASE1_RESULTS_DIR / 'best_feature_svm_metadata.joblib'
PHASE1_VALIDATION_PATH = PHASE1_RESULTS_DIR / 'feature_svm_validation_results.csv'
RESULTS_DIR = Path('log/exphase_2_result')
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

DEFAULT_TARGET_SIZE = (64, 64)  # cv2 uses (width, height)
DEFAULT_SVM_PARAMS = {
    'kernel': 'rbf',
    'C': 10.0,
    'gamma': 'scale',
    'class_weight': 'balanced',
    'cache_size': 1024,
}

phase1_metadata = {}
if PHASE1_METADATA_PATH.exists():
    phase1_metadata = joblib.load(PHASE1_METADATA_PATH)
    print(f'Loaded phase 1 metadata from {PHASE1_METADATA_PATH}')
else:
    print(f'Phase 1 metadata not found at {PHASE1_METADATA_PATH}; using defaults from phase 1 notebook.')

TARGET_SIZE = tuple(phase1_metadata.get('target_size', DEFAULT_TARGET_SIZE))
SVM_PARAMS = dict(phase1_metadata.get('svm_params', DEFAULT_SVM_PARAMS))

np.random.seed(RANDOM_STATE)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('Phase 2 config ready')
print('Target size:', TARGET_SIZE)
print('SVM params:', SVM_PARAMS)
print('Results dir:', RESULTS_DIR)

Loaded phase 1 metadata from log/exphase_1_result/best_feature_svm_metadata.joblib
Phase 2 config ready
Target size: (64, 64)
SVM params: {'kernel': 'rbf', 'C': 10.0, 'gamma': 'scale', 'class_weight': 'balanced', 'cache_size': 1024}
Results dir: log/exphase_2_result


/home/dacekey/AIL303_SUM26/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def read_image_bgr(path: Path):
    """Read image paths robustly, including class folders with non-ASCII names."""
    data = np.fromfile(str(path), dtype=np.uint8)
    image = cv2.imdecode(data, cv2.IMREAD_COLOR)
    return image


def resize_bgr(image_bgr: np.ndarray) -> np.ndarray:
    height, width = image_bgr.shape[:2]
    interpolation = cv2.INTER_AREA if height > TARGET_SIZE[1] or width > TARGET_SIZE[0] else cv2.INTER_CUBIC
    return cv2.resize(image_bgr, TARGET_SIZE, interpolation=interpolation)


def ensure_size_bgr(image_bgr: np.ndarray) -> np.ndarray:
    if image_bgr.shape[1] == TARGET_SIZE[0] and image_bgr.shape[0] == TARGET_SIZE[1]:
        return image_bgr
    return resize_bgr(image_bgr)


def get_class_names(dataset_dir: Path) -> list[str]:
    train_dir = dataset_dir / 'train'
    if not train_dir.exists():
        raise FileNotFoundError(f'Missing training directory: {train_dir}')
    return sorted(path.name for path in train_dir.iterdir() if path.is_dir())


def collect_split_paths(dataset_dir: Path, split: str, class_names: list[str]) -> tuple[list[Path], np.ndarray]:
    paths = []
    labels = []
    split_dir = dataset_dir / split
    if not split_dir.exists():
        raise FileNotFoundError(f'Missing split directory: {split_dir}')

    for label, class_name in enumerate(class_names):
        class_dir = split_dir / class_name
        if not class_dir.exists():
            print(f'Warning: missing class folder in {split}: {class_name}')
            continue
        class_paths = [p for p in sorted(class_dir.iterdir()) if p.suffix.lower() in IMAGE_EXTENSIONS]
        paths.extend(class_paths)
        labels.extend([label] * len(class_paths))
    return paths, np.asarray(labels, dtype=np.int64)


def load_split(dataset_dir: Path, split: str, class_names: list[str]) -> tuple[np.ndarray, np.ndarray, list[Path]]:
    paths, labels = collect_split_paths(dataset_dir, split, class_names)
    images = []
    bad_paths = []
    kept_labels = []
    kept_paths = []

    for path, label in tqdm(list(zip(paths, labels)), desc=f'Loading {split}'):
        image = read_image_bgr(path)
        if image is None:
            bad_paths.append(path)
            continue
        images.append(resize_bgr(image))
        kept_labels.append(label)
        kept_paths.append(path)

    if bad_paths:
        print(f'Skipped {len(bad_paths)} unreadable images from {split}')
    if not images:
        raise ValueError(f'No images loaded for split: {split}')

    return np.stack(images).astype(np.uint8), np.asarray(kept_labels, dtype=np.int64), kept_paths


def summarize_labels(y: np.ndarray, class_names: list[str], split: str) -> pd.DataFrame:
    counts = pd.Series(y).value_counts().sort_index()
    return pd.DataFrame({
        'split': split,
        'class_id': counts.index,
        'class_name': [class_names[i] for i in counts.index],
        'count': counts.values,
    })

In [3]:
folder_class_names = get_class_names(DATASET_DIR)
phase1_class_names = phase1_metadata.get('class_names')

if phase1_class_names is not None:
    class_names = list(phase1_class_names)
    missing_from_dataset = sorted(set(class_names) - set(folder_class_names))
    extra_in_dataset = sorted(set(folder_class_names) - set(class_names))
    if missing_from_dataset:
        raise ValueError(f'Phase 1 class names missing from dataset: {missing_from_dataset[:5]}')
    if extra_in_dataset:
        print(f'Warning: dataset has classes not in phase 1 metadata: {extra_in_dataset[:5]}')
    print('Using phase 1 label order from metadata.')
else:
    class_names = folder_class_names
    print('Using sorted train folder names as label order.')

label_to_name = dict(enumerate(class_names))

data = {}
for split in ['train', 'val', 'test']:
    images, labels, paths = load_split(DATASET_DIR, split, class_names)
    data[split] = {'images': images, 'y': labels, 'paths': paths}
    print(f'{split}: {images.shape[0]} images, image tensor shape={images.shape}')

label_summary = pd.concat(
    [summarize_labels(data[split]['y'], class_names, split) for split in ['train', 'val', 'test']],
    ignore_index=True,
)

display(label_summary.groupby('split')['count'].agg(['sum', 'min', 'median', 'max']))
display(label_summary.head())

Using phase 1 label order from metadata.


Loading train:   0%|          | 0/6605 [00:00<?, ?it/s]

Loading train:   6%|▌         | 396/6605 [00:00<00:01, 3958.25it/s]

Loading train:  13%|█▎        | 834/6605 [00:00<00:01, 4205.16it/s]

Loading train:  19%|█▉        | 1255/6605 [00:00<00:01, 4177.04it/s]

...

Loading test:  58%|█████▊    | 490/851 [00:00<00:00, 4892.21it/s]

Loading test: 100%|██████████| 851/851 [00:00<00:00, 4597.34it/s]

test: 851 images, image tensor shape=(851, 64, 64, 3)


,sum,min,median,max
split,,,,
test,851,3,11.0,108
train,6605,16,84.0,856
val,824,2,10.0,107


,split,class_id,class_name,count
0,train,0,B.8a,65
1,train,1,Camera,30
2,train,2,I.407a,156
3,train,3,I.409,20
4,train,4,I.434a,170


In [4]:
LBP_RADIUS = 1
LBP_N_POINTS = 8 * LBP_RADIUS
EDGE_GRID_SIZE = (4, 4)
CANNY_THRESHOLDS = (100, 200)
GABOR_ORIENTATIONS = [0, np.pi / 4, np.pi / 2, 3 * np.pi / 4]
GABOR_KERNEL_PARAMS = {
    'ksize': (21, 21),
    'sigma': 4.0,
    'lambd': 10.0,
    'gamma': 0.5,
    'psi': 0.0,
}


def to_gray_uint8(image_bgr: np.ndarray) -> np.ndarray:
    image_bgr = ensure_size_bgr(image_bgr)
    return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)


def to_gray_float32(image_bgr: np.ndarray) -> np.ndarray:
    return to_gray_uint8(image_bgr).astype(np.float32) / 255.0


def extract_lbp_feature(image_bgr: np.ndarray) -> np.ndarray:
    """Uniform LBP histogram with L1 normalization. Output dim: n_points + 2 = 10."""
    gray = to_gray_uint8(image_bgr)
    lbp = local_binary_pattern(gray, LBP_N_POINTS, LBP_RADIUS, method='uniform')
    n_bins = LBP_N_POINTS + 2
    hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, n_bins + 1))
    hist = hist.astype(np.float32)
    hist /= hist.sum() + 1e-8
    return hist


def extract_edge_histogram_feature(image_bgr: np.ndarray) -> np.ndarray:
    """Canny edge density in a 4x4 grid. Output dim: 16. No global density is appended."""
    gray = to_gray_uint8(image_bgr)
    edges = cv2.Canny(gray, CANNY_THRESHOLDS[0], CANNY_THRESHOLDS[1])
    height, width = edges.shape
    grid_rows, grid_cols = EDGE_GRID_SIZE
    cell_height = height // grid_rows
    cell_width = width // grid_cols

    densities = []
    for row in range(grid_rows):
        y0 = row * cell_height
        y1 = height if row == grid_rows - 1 else (row + 1) * cell_height
        for col in range(grid_cols):
            x0 = col * cell_width
            x1 = width if col == grid_cols - 1 else (col + 1) * cell_width
            cell = edges[y0:y1, x0:x1]
            densities.append(np.count_nonzero(cell) / float(cell.size))
    return np.asarray(densities, dtype=np.float32)


def extract_hu_moments_feature(image_bgr: np.ndarray) -> np.ndarray:
    """Hu moments from Otsu binary mask with signed log transform. Output dim: 7."""
    gray = to_gray_uint8(image_bgr)
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    moments = cv2.moments(binary)
    hu = cv2.HuMoments(moments).flatten()
    hu = -np.sign(hu) * np.log10(np.abs(hu) + 1e-12)
    return hu.astype(np.float32)


def extract_gabor_feature(image_bgr: np.ndarray) -> np.ndarray:
    """Mean and standard deviation of Gabor responses at four orientations. Output dim: 8."""
    gray = to_gray_float32(image_bgr)
    features = []
    for theta in GABOR_ORIENTATIONS:
        kernel = cv2.getGaborKernel(
            GABOR_KERNEL_PARAMS['ksize'],
            GABOR_KERNEL_PARAMS['sigma'],
            theta,
            GABOR_KERNEL_PARAMS['lambd'],
            GABOR_KERNEL_PARAMS['gamma'],
            GABOR_KERNEL_PARAMS['psi'],
            ktype=cv2.CV_32F,
        )
        kernel = kernel / (np.sum(np.abs(kernel)) + 1e-8)
        response = cv2.filter2D(gray, cv2.CV_32F, kernel)
        features.extend([float(response.mean()), float(response.std())])
    return np.asarray(features, dtype=np.float32)


def extract_feature_matrix(images_bgr: np.ndarray, extractor, desc: str) -> np.ndarray:
    features = [extractor(image) for image in tqdm(images_bgr, desc=desc)]
    return np.vstack(features).astype(np.float32)

In [5]:
FEATURE_LABELS = {
    'lbp': 'LBP',
    'edge_histogram': 'Edge Histogram',
    'hu_moments': 'Hu Moments',
    'gabor': 'Gabor',
}

FEATURE_EXTRACTORS = {
    'lbp': extract_lbp_feature,
    'edge_histogram': extract_edge_histogram_feature,
    'hu_moments': extract_hu_moments_feature,
    'gabor': extract_gabor_feature,
}

experiments = [
    {'case': 'LBP', 'feature_key': 'lbp'},
    {'case': 'Edge Histogram', 'feature_key': 'edge_histogram'},
    {'case': 'Hu Moments', 'feature_key': 'hu_moments'},
    {'case': 'Gabor', 'feature_key': 'gabor'},
]

FEATURE_CACHE = {}
FEATURE_TIMINGS = {}


def get_feature_matrix(split: str, feature_key: str) -> np.ndarray:
    cache_key = (split, feature_key)
    if cache_key in FEATURE_CACHE:
        return FEATURE_CACHE[cache_key]

    images = data[split]['images']
    extractor = FEATURE_EXTRACTORS[feature_key]
    started = perf_counter()
    matrix = extract_feature_matrix(images, extractor, desc=f'{split}/{FEATURE_LABELS[feature_key]}')
    elapsed = perf_counter() - started

    FEATURE_CACHE[cache_key] = matrix.astype(np.float32)
    FEATURE_TIMINGS[cache_key] = elapsed
    print(f'{split}/{feature_key}: {FEATURE_CACHE[cache_key].shape}, feature_time_sec={elapsed:.3f}')
    return FEATURE_CACHE[cache_key]


def feature_name(feature_key: str) -> str:
    return FEATURE_LABELS[feature_key]


pd.DataFrame({
    'case': [experiment['case'] for experiment in experiments],
    'features': [feature_name(experiment['feature_key']) for experiment in experiments],
})

,case,features
0,LBP,LBP
1,Edge Histogram,Edge Histogram
2,Hu Moments,Hu Moments
3,Gabor,Gabor


In [6]:
def build_svm_pipeline():
    return make_pipeline(
        StandardScaler(),
        SVC(**SVM_PARAMS),
    )


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'macro_precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'macro_recall': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'weighted_f1': f1_score(y_true, y_pred, average='weighted', zero_division=0),
    }


def slugify(text: str) -> str:
    chars = []
    for char in text.lower():
        if char.isalnum():
            chars.append(char)
        else:
            chars.append('_')
    return '_'.join(''.join(chars).split('_')).strip('_')


def top_confusions(y_true: np.ndarray, y_pred: np.ndarray, class_names: list[str], top_n: int = 20) -> pd.DataFrame:
    columns = ['true_class', 'predicted_class', 'count']
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(class_names)))
    rows = []
    for true_id in range(cm.shape[0]):
        for pred_id in range(cm.shape[1]):
            if true_id == pred_id or cm[true_id, pred_id] == 0:
                continue
            rows.append({
                'true_class': class_names[true_id],
                'predicted_class': class_names[pred_id],
                'count': int(cm[true_id, pred_id]),
            })
    if not rows:
        return pd.DataFrame(columns=columns)
    return pd.DataFrame(rows, columns=columns).sort_values('count', ascending=False).head(top_n)


def save_classification_outputs(
    split_name: str,
    case: str,
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> dict[str, Path]:
    slug = slugify(case)
    report_dict = classification_report(
        y_true,
        y_pred,
        labels=np.arange(len(class_names)),
        target_names=class_names,
        zero_division=0,
        output_dict=True,
    )
    report_text = classification_report(
        y_true,
        y_pred,
        labels=np.arange(len(class_names)),
        target_names=class_names,
        zero_division=0,
    )
    report_df = pd.DataFrame(report_dict).transpose()
    report_csv_path = RESULTS_DIR / f'{split_name}_classification_report_{slug}.csv'
    report_txt_path = RESULTS_DIR / f'{split_name}_classification_report_{slug}.txt'
    report_df.to_csv(report_csv_path)
    report_txt_path.write_text(report_text, encoding='utf-8')

    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(class_names)))
    cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
    cm_path = RESULTS_DIR / f'{split_name}_confusion_matrix_{slug}.csv'
    cm_df.to_csv(cm_path, index_label='true_class')

    top_confusions_df = top_confusions(y_true, y_pred, class_names)
    top_confusions_path = RESULTS_DIR / f'{split_name}_top_confusions_{slug}.csv'
    top_confusions_df.to_csv(top_confusions_path, index=False)

    return {
        'classification_report_csv': report_csv_path,
        'classification_report_txt': report_txt_path,
        'confusion_matrix_csv': cm_path,
        'top_confusions_csv': top_confusions_path,
    }

In [7]:
validation_results = []
summary_results = []
trained_models = {}

for experiment in experiments:
    case = experiment['case']
    feature_key = experiment['feature_key']
    print(f"\nRunning: {case}")

    X_train = get_feature_matrix('train', feature_key)
    X_val = get_feature_matrix('val', feature_key)
    y_train = data['train']['y']
    y_val = data['val']['y']

    model = build_svm_pipeline()
    train_started = perf_counter()
    model.fit(X_train, y_train)
    train_time_sec = perf_counter() - train_started

    predict_started = perf_counter()
    val_pred = model.predict(X_val)
    predict_time_sec = perf_counter() - predict_started

    val_feature_time_sec = FEATURE_TIMINGS[('val', feature_key)]
    inference_time_ms_per_image = ((val_feature_time_sec + predict_time_sec) / len(y_val)) * 1000.0
    predict_time_ms_per_image = (predict_time_sec / len(y_val)) * 1000.0
    metrics = compute_metrics(y_val, val_pred)

    output_paths = save_classification_outputs('val', case, y_val, val_pred)
    trained_models[case] = model

    validation_results.append({
        'case': case,
        'features': feature_name(feature_key),
        'feature_keys': feature_key,
        'n_features': X_train.shape[1],
        'train_seconds': train_time_sec,
        'feature_extraction_train_sec': FEATURE_TIMINGS[('train', feature_key)],
        'feature_extraction_val_sec': val_feature_time_sec,
        'val_predict_seconds': predict_time_sec,
        'val_inference_time_ms_per_image': inference_time_ms_per_image,
        'val_predict_time_ms_per_image': predict_time_ms_per_image,
        **{f'val_{key}': value for key, value in metrics.items()},
        **{key: str(path) for key, path in output_paths.items()},
    })

    summary_results.append({
        'case': case,
        'features': feature_name(feature_key),
        'feature_keys': feature_key,
        'n_features': X_train.shape[1],
        'eval_split': 'val',
        'accuracy': metrics['accuracy'],
        'macro_precision': metrics['macro_precision'],
        'macro_recall': metrics['macro_recall'],
        'macro_f1': metrics['macro_f1'],
        'weighted_f1': metrics['weighted_f1'],
        'train_time_sec': train_time_sec,
        'inference_time_ms_per_image': inference_time_ms_per_image,
        'predict_time_ms_per_image': predict_time_ms_per_image,
    })

    print(
        f"macro_f1={metrics['macro_f1']:.4f}, "
        f"accuracy={metrics['accuracy']:.4f}, "
        f"n_features={X_train.shape[1]}, "
        f"train_time_sec={train_time_sec:.1f}, "
        f"inference_time_ms_per_image={inference_time_ms_per_image:.3f}"
    )

validation_df = pd.DataFrame(validation_results).sort_values('val_macro_f1', ascending=False).reset_index(drop=True)
phase2_results_df = pd.DataFrame(summary_results).sort_values('macro_f1', ascending=False).reset_index(drop=True)

validation_path = RESULTS_DIR / 'feature_svm_validation_results.csv'
phase2_results_path = RESULTS_DIR / 'feature_svm_phase2_results.csv'
validation_df.to_csv(validation_path, index=False)
phase2_results_df.to_csv(phase2_results_path, index=False)

print(f'Saved validation results to {validation_path}')
print(f'Saved phase 2 summary results to {phase2_results_path}')
print('Phase 2 validation results sorted by macro_f1 descending:')
display(phase2_results_df)


Running: LBP


train/LBP:   0%|          | 0/6605 [00:00<?, ?it/s]

train/LBP:   2%|▏         | 143/6605 [00:00<00:04, 1429.20it/s]

train/LBP:   4%|▍         | 287/6605 [00:00<00:04, 1433.80it/s]

train/LBP:   7%|▋         | 432/6605 [00:00<00:04, 1438.79it/s]

...

val/Gabor:  87%|████████▋ | 717/824 [00:00<00:00, 1434.40it/s]

val/Gabor: 100%|██████████| 824/824 [00:00<00:00, 1430.01it/s]

val/gabor: (824, 8), feature_time_sec=0.579


macro_f1=0.5874, accuracy=0.5934, n_features=8, train_time_sec=0.8, inference_time_ms_per_image=1.065
Saved validation results to log/exphase_2_result/feature_svm_validation_results.csv
Saved phase 2 summary results to log/exphase_2_result/feature_svm_phase2_results.csv
Phase 2 validation results sorted by macro_f1 descending:


,case,features,feature_keys,n_features,eval_split,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,train_time_sec,inference_time_ms_per_image,predict_time_ms_per_image
0,Edge Histogram,Edge Histogram,edge_histogram,16,val,0.641990,0.607515,0.609466,0.587502,0.640616,0.883584,0.518051,0.403818
1,Gabor,Gabor,gabor,8,val,0.593447,0.568117,0.699511,0.587437,0.587503,0.765828,1.065085,0.362361
2,LBP,LBP,lbp,10,val,0.379854,0.318934,0.372002,0.307363,0.394689,1.072376,1.153238,0.428231
3,Hu Moments,Hu Moments,hu_moments,7,val,0.314320,0.261156,0.365685,0.251126,0.320610,1.189613,0.439222,0.408797


In [8]:
if PHASE1_VALIDATION_PATH.exists():
    phase1_validation_df = pd.read_csv(PHASE1_VALIDATION_PATH)
    phase1_validation_df.insert(0, 'phase', 'phase_1')

    phase2_compare_df = validation_df.copy()
    phase2_compare_df.insert(0, 'phase', 'phase_2')

    combined_validation_df = pd.concat(
        [phase1_validation_df, phase2_compare_df],
        ignore_index=True,
        sort=False,
    ).sort_values('val_macro_f1', ascending=False).reset_index(drop=True)

    combined_path = RESULTS_DIR / 'feature_svm_phase1_phase2_validation_comparison.csv'
    combined_validation_df.to_csv(combined_path, index=False)
    print(f'Saved combined phase 1 + phase 2 validation comparison to {combined_path}')
    display(combined_validation_df[['phase', 'case', 'features', 'n_features', 'train_seconds', 'val_accuracy', 'val_macro_f1', 'val_weighted_f1']].head(20))
else:
    print(f'Phase 1 validation CSV not found at {PHASE1_VALIDATION_PATH}; skipped combined comparison.')

Saved combined phase 1 + phase 2 validation comparison to log/exphase_2_result/feature_svm_phase1_phase2_validation_comparison.csv


,phase,case,features,n_features,train_seconds,val_accuracy,val_macro_f1,val_weighted_f1
0,phase_1,Raw Pixels (Baseline),Raw Pixels (gray),4096,59.909299,0.942961,0.934047,0.942376
1,phase_1,HOG + Color Histogram - gray,HOG (gray) + Color Histogram (HSV),2276,40.240549,0.955097,0.932733,0.953546
2,phase_1,HOG only - gray,HOG (gray),1764,29.588671,0.952670,0.928592,0.950699
3,phase_1,HOG only - yuv,HOG (YUV),5292,135.790715,0.955097,0.927394,0.952822
4,phase_1,HOG + Color Histogram - yuv,HOG (YUV) + Color Histogram (HSV),5804,142.904578,0.955097,0.920911,0.952630
5,phase_1,HOG only - clahe,HOG (CLAHE),1764,29.747546,0.941748,0.920812,0.939684
6,phase_1,HOG + Color Histogram - clahe,HOG (CLAHE) + Color Histogram (HSV),2276,41.565585,0.942961,0.913737,0.940918
7,phase_1,Color Histogram,Color Histogram (HSV),512,3.177264,0.907767,0.906096,0.908443
8,phase_1,HOG + Color Histogram - hue,HOG (Hue) + Color Histogram (HSV),2276,68.407465,0.879854,0.798609,0.872120
9,phase_1,HOG only - hue,HOG (Hue),1764,48.210326,0.765777,0.625645,0.753311


In [9]:
best_validation_row = validation_df.iloc[0]
best_case = best_validation_row['case']
best_feature_key = best_validation_row['feature_keys']
best_model = trained_models[best_case]

print('Best validation experiment for phase 2')
display(best_validation_row.to_frame(name='value'))

X_test = get_feature_matrix('test', best_feature_key)
y_test = data['test']['y']
test_predict_started = perf_counter()
test_pred = best_model.predict(X_test)
test_predict_time_sec = perf_counter() - test_predict_started
test_feature_time_sec = FEATURE_TIMINGS[('test', best_feature_key)]
test_inference_time_ms_per_image = ((test_feature_time_sec + test_predict_time_sec) / len(y_test)) * 1000.0
test_predict_time_ms_per_image = (test_predict_time_sec / len(y_test)) * 1000.0
test_metrics = compute_metrics(y_test, test_pred)

best_test_metrics_df = pd.DataFrame([{
    'case': best_case,
    'features': feature_name(best_feature_key),
    'feature_keys': best_feature_key,
    'n_features': X_test.shape[1],
    **{f'test_{key}': value for key, value in test_metrics.items()},
    'test_feature_extraction_sec': test_feature_time_sec,
    'test_predict_seconds': test_predict_time_sec,
    'test_inference_time_ms_per_image': test_inference_time_ms_per_image,
    'test_predict_time_ms_per_image': test_predict_time_ms_per_image,
}])

best_test_summary_df = pd.DataFrame([{
    'case': best_case,
    'features': feature_name(best_feature_key),
    'feature_keys': best_feature_key,
    'n_features': X_test.shape[1],
    'eval_split': 'test',
    'accuracy': test_metrics['accuracy'],
    'macro_precision': test_metrics['macro_precision'],
    'macro_recall': test_metrics['macro_recall'],
    'macro_f1': test_metrics['macro_f1'],
    'weighted_f1': test_metrics['weighted_f1'],
    'train_time_sec': best_validation_row['train_seconds'],
    'inference_time_ms_per_image': test_inference_time_ms_per_image,
    'predict_time_ms_per_image': test_predict_time_ms_per_image,
}])

best_test_metrics_path = RESULTS_DIR / 'feature_svm_best_test_metrics.csv'
best_test_summary_path = RESULTS_DIR / 'feature_svm_phase2_best_test_results.csv'
best_test_metrics_df.to_csv(best_test_metrics_path, index=False)
best_test_summary_df.to_csv(best_test_summary_path, index=False)
test_output_paths = save_classification_outputs('test', best_case, y_test, test_pred)

print(f'Saved best test metrics to {best_test_metrics_path}')
print(f'Saved best test summary to {best_test_summary_path}')
print(f'Saved best test classification outputs: {test_output_paths}')
display(best_test_summary_df)

Best validation experiment for phase 2


,value
case,Edge Histogram
features,Edge Histogram
feature_keys,edge_histogram
n_features,16
train_seconds,0.883584
feature_extraction_train_sec,0.73075
feature_extraction_val_sec,0.094128
val_predict_seconds,0.332746
val_inference_time_ms_per_image,0.518051
val_predict_time_ms_per_image,0.403818


test/Edge Histogram:   0%|          | 0/851 [00:00<?, ?it/s]

test/Edge Histogram: 100%|██████████| 851/851 [00:00<00:00, 9232.32it/s]

test/edge_histogram: (851, 16), feature_time_sec=0.094


Saved best test metrics to log/exphase_2_result/feature_svm_best_test_metrics.csv
Saved best test summary to log/exphase_2_result/feature_svm_phase2_best_test_results.csv
Saved best test classification outputs: {'classification_report_csv': PosixPath('log/exphase_2_result/test_classification_report_edge_histogram.csv'), 'classification_report_txt': PosixPath('log/exphase_2_result/test_classification_report_edge_histogram.txt'), 'confusion_matrix_csv': PosixPath('log/exphase_2_result/test_confusion_matrix_edge_histogram.csv'), 'top_confusions_csv': PosixPath('log/exphase_2_result/test_top_confusions_edge_histogram.csv')}


,case,features,feature_keys,n_features,eval_split,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,train_time_sec,inference_time_ms_per_image,predict_time_ms_per_image
0,Edge Histogram,Edge Histogram,edge_histogram,16,test,0.614571,0.62739,0.590094,0.592619,0.618852,0.883584,0.511224,0.400368


In [10]:
best_model_path = RESULTS_DIR / 'best_feature_svm_pipeline.joblib'
metadata_path = RESULTS_DIR / 'best_feature_svm_metadata.joblib'

joblib.dump(best_model, best_model_path)
joblib.dump(
    {
        'phase': 'phase_2',
        'best_case': best_case,
        'best_feature_key': best_feature_key,
        'class_names': class_names,
        'target_size': TARGET_SIZE,
        'svm_params': SVM_PARAMS,
        'random_state': RANDOM_STATE,
        'feature_params': {
            'lbp': {
                'radius': LBP_RADIUS,
                'n_points': LBP_N_POINTS,
                'method': 'uniform',
                'histogram_normalization': 'L1',
            },
            'edge_histogram': {
                'method': 'cv2.Canny',
                'threshold1': CANNY_THRESHOLDS[0],
                'threshold2': CANNY_THRESHOLDS[1],
                'grid_size': EDGE_GRID_SIZE,
                'feature_dim': 16,
                'global_density_appended': False,
            },
            'hu_moments': {
                'threshold': 'Otsu',
                'binary_inversion': False,
                'log_transform': '-sign(x) * log10(abs(x) + 1e-12)',
                'feature_dim': 7,
            },
            'gabor': {
                'orientations': GABOR_ORIENTATIONS,
                **GABOR_KERNEL_PARAMS,
                'summary_stats': ['mean', 'std'],
                'feature_dim': 8,
            },
        },
        'phase1_metadata_path': str(PHASE1_METADATA_PATH),
        'phase1_validation_path': str(PHASE1_VALIDATION_PATH),
    },
    metadata_path,
)

print(f'Saved best model to {best_model_path}')
print(f'Saved metadata to {metadata_path}')

Saved best model to log/exphase_2_result/best_feature_svm_pipeline.joblib
Saved metadata to log/exphase_2_result/best_feature_svm_metadata.joblib
